In [8]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [10]:
def process_all_txts(txt_directory):
    """Process all TXT files in a directory"""
    all_documents = []
    txt_dir = Path(txt_directory)
    
    # Find all TXT files recursively
    txt_files = list(txt_dir.glob("**/*.txt"))
    
    print(f"Found {len(txt_files)} TXT files to process")
    
    for txt_file in txt_files:
        print(f"\nProcessing: {txt_file.name}")
        try:
            with open(txt_file, "r", encoding="utf-8") as f:
                content = f.read()
                doc = {
                    "content": content,
                    "metadata": {
                        "source_file": txt_file.name,
                        "file_type": "txt"
                    }
                }
                all_documents.append(doc)
            print(f"  ✓ Loaded {len(content.splitlines())} lines")
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all TXT files in the data directory
all_txt_documents = process_all_txts("../data")

Found 56 TXT files to process

Processing: ADMIN_OPERATIONS_API_DEEP_DIVE.txt
  ✓ Loaded 2 lines

Processing: ADMIN_OPERATIONS_DEEP_DIVE.txt
  ✓ Loaded 145 lines

Processing: ANALYTICS_REPORTING_API_DEEP_DIVE.txt
  ✓ Loaded 2 lines

Processing: ANALYTICS_REPORTING_DEEP_DIVE.txt
  ✓ Loaded 148 lines

Processing: APPENDIX_EXAMPLE_PAYLOADS.txt
  ✓ Loaded 236 lines

Processing: CHATBOT_ADVANCED_PATTERNS.txt
  ✓ Loaded 109 lines

Processing: CHATBOT_ADVANCED_USE_CASES.txt
  ✓ Loaded 2 lines

Processing: CHATBOT_ARCHITECTURE_DEEP_DIVE.txt
  ✓ Loaded 99 lines

Processing: CHATBOT_CODE_EXAMPLES.txt
  ✓ Loaded 2 lines

Processing: CHATBOT_DEVELOPER_REFERENCE.txt
  ✓ Loaded 2 lines

Processing: CHATBOT_FEATURE_USER_GUIDE.txt
  ✓ Loaded 114 lines

Processing: CHATBOT_SECURITY_AUDIT_CHECKLIST.txt
  ✓ Loaded 2 lines

Processing: CHATBOT_SECURITY_BEST_PRACTICES.txt
  ✓ Loaded 2 lines

Processing: CHATBOT_SECURITY_REMEDIATION_REPORT.txt
  ✓ Loaded 77 lines

Processing: CHATBOT_SECURITY_REMEDIATION_RE

In [11]:
all_txt_documents

[{'content': '(Moved from backend/DEMP/ADMIN_OPERATIONS_API_DEEP_DIVE.txt)\n\n',
  'metadata': {'source_file': 'ADMIN_OPERATIONS_API_DEEP_DIVE.txt',
   'file_type': 'txt'}},
 {'content': '# ADMIN OPERATIONS DEEP DIVE\n\nThis document provides a comprehensive technical and operational deep dive into the Admin Operations module of the Digital Event Management Platform (DEMP). It covers architecture, workflows, API endpoints, validation, security, extensibility, and integration best practices.\n\n---\n\n## 1. Overview\n- **Purpose:** Enable platform administrators to manage, monitor, and configure all aspects of the platform, ensuring security, compliance, and operational excellence.\n- **Key Features:**\n  - User and role management\n  - Event and ticketing oversight\n  - Payment and refund monitoring\n  - System configuration and settings\n  - Audit logging and compliance\n  - Platform health monitoring and alerts\n\n---\n\n## 2. Admin Workflows\n- **User Management:**\n  - View, edit, 

In [12]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    # Convert dicts to Document objects for compatibility
    from langchain_core.documents import Document
    doc_objs = [Document(page_content=doc['content'], metadata=doc['metadata']) for doc in documents]
    split_docs = text_splitter.split_documents(doc_objs)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [13]:
chunks=split_documents(all_txt_documents)
chunks

Split 56 documents into 255 chunks

Example chunk:
Content: (Moved from backend/DEMP/ADMIN_OPERATIONS_API_DEEP_DIVE.txt)...
Metadata: {'source_file': 'ADMIN_OPERATIONS_API_DEEP_DIVE.txt', 'file_type': 'txt'}


[Document(metadata={'source_file': 'ADMIN_OPERATIONS_API_DEEP_DIVE.txt', 'file_type': 'txt'}, page_content='(Moved from backend/DEMP/ADMIN_OPERATIONS_API_DEEP_DIVE.txt)'),
 Document(metadata={'source_file': 'ADMIN_OPERATIONS_DEEP_DIVE.txt', 'file_type': 'txt'}, page_content='# ADMIN OPERATIONS DEEP DIVE\n\nThis document provides a comprehensive technical and operational deep dive into the Admin Operations module of the Digital Event Management Platform (DEMP). It covers architecture, workflows, API endpoints, validation, security, extensibility, and integration best practices.\n\n---\n\n## 1. Overview\n- **Purpose:** Enable platform administrators to manage, monitor, and configure all aspects of the platform, ensuring security, compliance, and operational excellence.\n- **Key Features:**\n  - User and role management\n  - Event and ticketing oversight\n  - Payment and refund monitoring\n  - System configuration and settings\n  - Audit logging and compliance\n  - Platform health monitor

In [14]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1563.20it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\PA40139632\AppData\Local\Temp\ipykernel_36140\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [16]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "txt_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store for TXT documents
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "TXT document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: txt_documents
Existing documents in collection: 23


In [17]:
# Prepare the texts from your chunks
texts = [chunk.page_content for chunk in chunks]

# Generate embeddings for each chunk
embeddings = embedding_manager.generate_embeddings(texts)

# Add the chunks and their embeddings to the vector store
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 255 texts...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches: 100%|██████████| 8/8 [00:13<00:00,  1.69s/it]


Generated embeddings with shape: (255, 384)
Adding 255 documents to vector store...
Successfully added 255 documents to vector store
Total documents in collection: 278


In [18]:
print("Total documents in collection:", vectorstore.collection.count())

Total documents in collection: 278


In [19]:
chunks

[Document(metadata={'source_file': 'ADMIN_OPERATIONS_API_DEEP_DIVE.txt', 'file_type': 'txt'}, page_content='(Moved from backend/DEMP/ADMIN_OPERATIONS_API_DEEP_DIVE.txt)'),
 Document(metadata={'source_file': 'ADMIN_OPERATIONS_DEEP_DIVE.txt', 'file_type': 'txt'}, page_content='# ADMIN OPERATIONS DEEP DIVE\n\nThis document provides a comprehensive technical and operational deep dive into the Admin Operations module of the Digital Event Management Platform (DEMP). It covers architecture, workflows, API endpoints, validation, security, extensibility, and integration best practices.\n\n---\n\n## 1. Overview\n- **Purpose:** Enable platform administrators to manage, monitor, and configure all aspects of the platform, ensuring security, compliance, and operational excellence.\n- **Key Features:**\n  - User and role management\n  - Event and ticketing oversight\n  - Payment and refund monitoring\n  - System configuration and settings\n  - Audit logging and compliance\n  - Platform health monitor

In [20]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [21]:
rag_retriever

In [22]:
rag_retriever.retrieve("What key features of python programming language")

Retrieving documents for query: 'What key features of python programming language'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.99it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_72ddeb86_4',
  'content': 'Introduction to Python\n\nPython is a high-level, interpreted programming language known for its simplicity, readability, and versatility. Created by Guido van Rossum and first released in 1991, Python has become one of the most popular programming languages in the world.\n\n---\n\nHistory:\nPython was conceived in the late 1980s as a successor to the ABC language. Its first public release was in 1991. Over the years, Python has evolved through several major versions, with Python 3 being the current standard.\n\nKey Features:\n- Easy-to-read syntax that emphasizes code readability\n- Dynamically typed and interpreted, making it beginner-friendly\n- Supports multiple programming paradigms: procedural, object-oriented, and functional\n- Extensive standard library and a large ecosystem of third-party packages\n- Cross-platform compatibility (Windows, macOS, Linux)\n- Large and active community',
  'metadata': {'content_length': 888,
   'file_type': 

In [23]:
import json
 
import requests
 
def ollama_rag_response(
    query,
    rag_retriever,
    ollama_url="http://localhost:11434/api/generate",
    model_name="qwen3.5:2b",
    top_k=5
):
    # Step 1: Retrieve relevant context from vector DB
    retrieved_docs = rag_retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in retrieved_docs]) if retrieved_docs else ""
    if not context:
        return "No relevant context found to answer the question."
 
    # Step 2: Prepare prompt for LLM
    print("calling llm")
    prompt = f"""Context:\n{context}\n\nQuestion: {query}\nAnswer:"""
    print("loading data")
    # Step 3: Call Ollama LLM API (streaming)
    payload = {
        "model": model_name,
        "prompt": prompt
    }
    print("payload prepared, making API call...")
    response = requests.post(ollama_url, json=payload, stream=True)
    response.raise_for_status()
    answer = ""
    for line in response.iter_lines():
        if line:
            try:
                data = json.loads(line.decode("utf-8"))
                answer += data.get("response", "")
            except Exception:
                continue
    return answer if answer else "No response from LLM."
 

In [26]:
question = "what are Analytics Lifecycle & Workflows"
response = ollama_rag_response(question, rag_retriever, model_name="qwen3.5:2b")
print(response)

Retrieving documents for query: 'what are Analytics Lifecycle & Workflows'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:03<00:00,  3.79s/it]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
calling llm
loading data
payload prepared, making API call...
Based on the provided context, the **Analytics Lifecycle & Workflows** are defined through the following operational phases:

*   **Data Collection:**
    *   Ingesting data from all platform modules (events, users, tickets, payments, media, notifications).
    *   Performing **ETL (Extract, Transform, Load)** processes for data normalization.

*   **Processing:**
    *   Aggregate, filter, and compute metrics (including attendance, revenue, and engagement).
    *   Store the normalized data in an analytics-optimized data store (such as a data warehouse or OLAP system).

*   **Visualization:**
    *   Create dashboards for specific stakeholders:
        *   **Organizers:** Event performance, ticket sales, revenue.
        *   **Admins:** Platform usage, user growth, financials.
        *   **Users:** Personal activity and ticket history.

*   *